# 🤖 ТАРС v4 — Автономное обучение

**Всё через один скрипт `local_train.py`. Всё сохраняется на Google Drive. Переподключение = продолжение.**

| Что | Где |
|---|---|
| Датасеты | `MyDrive/TarsData/` |
| Модели + чекпоинты | `MyDrive/TarsModels/` |
| Логи | `local_train.log` |

### Уровни обучения
| Уровень | Время | Назначение |
|---|---|---|
| `small` | ~15 мин | Быстрый тест, отладка |
| `medium` | ~3 часа | Стандартное обучение |
| `max` | ~15 часов | Продакшн |
| `marathon` | ~96 часов | 4 дня непрерывно, максимальное качество |

### Инструкция
1. **Runtime → Change runtime type → L4 GPU** (или T4 бесплатно)
2. **Выберите уровень** в ячейке ниже и запустите
3. Если отключилось — просто **запустить ячейку заново** (всё продолжится с места остановки)

In [ ]:
#@title 🚀 **ОБУЧЕНИЕ ТАРС** — всё через один скрипт { display-mode: "form" }

#@markdown ### Настройки
LEVEL = "medium" #@param ["small", "medium", "max", "marathon"] {type:"string"}
#@markdown - `small` — быстрый тест (~15 мин)
#@markdown - `medium` — стандарт (~3 часа)
#@markdown - `max` — продакшн (~15 часов)
#@markdown - `marathon` — 4 дня непрерывно (~96ч, макс. качество)

SKIP_VOICE = True #@param {type:"boolean"}
SKIP_POSTTRAIN = False #@param {type:"boolean"}

# ============================================================
import os, sys, time, shutil, subprocess
from pathlib import Path

# ==== 1. GOOGLE DRIVE ====
print('=' * 60)
print('  \ud83e\udd16 \u0422\u0410\u0420\u0421 v4 \u2014 \u0410\u0432\u0442\u043e\u043d\u043e\u043c\u043d\u043e\u0435 \u043e\u0431\u0443\u0447\u0435\u043d\u0438\u0435')
print('=' * 60)
print()

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA   = Path('/content/drive/MyDrive/TarsData')
DRIVE_MODELS = Path('/content/drive/MyDrive/TarsModels')
DRIVE_DATA.mkdir(parents=True, exist_ok=True)
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)

print(f'  \u2601\ufe0f  Drive: data={DRIVE_DATA}')
print(f'           models={DRIVE_MODELS}')

existing_data = list(DRIVE_DATA.glob('*.txt'))
existing_models = list(DRIVE_MODELS.glob('*.pt'))
if existing_data:
    mb = sum(f.stat().st_size for f in existing_data) / 1024**2
    print(f'  \ud83d\udcc2 \u0414\u0430\u043d\u043d\u044b\u0435: {len(existing_data)} \u0444\u0430\u0439\u043b\u043e\u0432, {mb:.0f} MB')
if existing_models:
    mb = sum(f.stat().st_size for f in existing_models) / 1024**2
    print(f'  \ud83e\udde0 \u041c\u043e\u0434\u0435\u043b\u0438: {len(existing_models)} \u0444\u0430\u0439\u043b\u043e\u0432, {mb:.0f} MB')
print()

# ==== 2. GPU CHECK ====
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print()

# ==== 3. CLONE / UPDATE CODE ====
os.chdir('/content')
if os.path.exists('TarsSSM-Py'):
    os.chdir('/content/TarsSSM-Py')
    !git pull --rebase 2>/dev/null || true
else:
    !git clone https://github.com/Vazilll/TarsSSM-Py.git
    os.chdir('/content/TarsSSM-Py')

ROOT = Path('/content/TarsSSM-Py')
print(f'  \ud83d\udcc1 Root: {ROOT}')

# ==== 4. SYMLINKS: data/ \u2192 Drive, models/ \u2192 Drive ====
def safe_symlink(local, drive_target):
    if local.is_symlink():
        return
    if local.exists():
        for f in local.rglob('*'):
            if f.is_file():
                rel = f.relative_to(local)
                dest = drive_target / rel
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists():
                    shutil.move(str(f), str(dest))
        shutil.rmtree(str(local))
    local.symlink_to(drive_target)

safe_symlink(ROOT / 'data', DRIVE_DATA)
safe_symlink(ROOT / 'models', DRIVE_MODELS)
print('  \ud83d\udd17 data/ \u2192 Drive/TarsData')
print('  \ud83d\udd17 models/ \u2192 Drive/TarsModels')
print()

# ==== 5. INSTALL DEPS ====
print('  \ud83d\udce6 \u0423\u0441\u0442\u0430\u043d\u043e\u0432\u043a\u0430 \u0437\u0430\u0432\u0438\u0441\u0438\u043c\u043e\u0441\u0442\u0435\u0439...')
!pip install -q torch einops numpy tqdm sentencepiece datasets psutil 2>/dev/null
print()

# ==== 6. KEEP-ALIVE (\u0430\u043d\u0442\u0438-\u0442\u0430\u0439\u043c\u0430\u0443\u0442) ====
import threading
def _keep_alive():
    while True:
        time.sleep(600)
        elapsed = time.time() - _train_start
        h, m = int(elapsed // 3600), int((elapsed % 3600) // 60)
        print(f'  \u23f0 Keep-alive: {h}h {m}m \u2014 \u043e\u0431\u0443\u0447\u0435\u043d\u0438\u0435 \u043f\u0440\u043e\u0434\u043e\u043b\u0436\u0430\u0435\u0442\u0441\u044f...', flush=True)

_train_start = time.time()
_ka = threading.Thread(target=_keep_alive, daemon=True)
_ka.start()

# ==== 7. CHECKPOINT STATE ====
has_data = len(list(DRIVE_DATA.glob('hf_*.txt'))) > 0 or len(list(DRIVE_DATA.glob('wiki_*.txt'))) > 0

extra_args = []
if has_data:
    extra_args.append('--skip-download')
    print('  \u2705 \u0414\u0430\u043d\u043d\u044b\u0435 \u043d\u0430 Drive \u2014 \u043f\u0440\u043e\u043f\u0443\u0441\u043a \u0441\u043a\u0430\u0447\u0438\u0432\u0430\u043d\u0438\u044f')

extra_args += ['--resume']  # \u0412\u0441\u0435\u0433\u0434\u0430 --resume \u0434\u043b\u044f Colab (\u043c\u043e\u0433\u0443\u0442 \u043e\u0442\u043a\u043b\u044e\u0447\u0438\u0442\u044c)

if SKIP_VOICE:
    extra_args.append('--skip-voice')
if SKIP_POSTTRAIN:
    extra_args.append('--skip-posttrain')
print()

# ==== 8. \u041e\u0411\u0423\u0427\u0415\u041d\u0418\u0415 \u0447\u0435\u0440\u0435\u0437 local_train.py ====
print('=' * 60)
print(f'  \ud83c\udf93 \u041d\u0430\u0447\u0430\u043b\u043e \u043e\u0431\u0443\u0447\u0435\u043d\u0438\u044f: \u0443\u0440\u043e\u0432\u0435\u043d\u044c={LEVEL}')
print('  \u0412\u0441\u0451 \u0441\u043e\u0445\u0440\u0430\u043d\u044f\u0435\u0442\u0441\u044f \u043d\u0430 Drive \u0430\u0432\u0442\u043e\u043c\u0430\u0442\u0438\u0447\u0435\u0441\u043a\u0438')
print('=' * 60)
print(flush=True)

# \u041c\u0430\u043f\u043f\u0438\u043d\u0433 \u0443\u0440\u043e\u0432\u043d\u0435\u0439 (\u0435\u0441\u043b\u0438 \u043f\u043e\u043b\u044c\u0437\u043e\u0432\u0430\u0442\u0435\u043b\u044c \u0432\u044b\u0431\u0440\u0430\u043b small)
level_map = {'small': 'small'}
actual_level = level_map.get(LEVEL, LEVEL)

cmd = [
    sys.executable, '-u', str(ROOT / 'local_train.py'),
    '--level', actual_level,
    '--drive', 'colab',
] + extra_args

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

process = subprocess.Popen(
    cmd,
    cwd=str(ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=env,
    bufsize=1,
    universal_newlines=True,
)

for line in process.stdout:
    print(line, end='', flush=True)

returncode = process.wait()

# ==== 9. \u0420\u0415\u0417\u0423\u041b\u042c\u0422\u0410\u0422 ====
elapsed = time.time() - _train_start
hours = elapsed / 3600
minutes = elapsed / 60

print()
print('=' * 60)
if returncode == 0:
    print(f'  \u2705 \u041e\u0411\u0423\u0427\u0415\u041d\u0418\u0415 \u0417\u0410\u0412\u0415\u0420\u0428\u0415\u041d\u041e \u0437\u0430 {hours:.1f}\u0447 ({minutes:.0f} \u043c\u0438\u043d)!')
    print()
    for f in sorted(DRIVE_MODELS.rglob('*.pt')):
        mb = f.stat().st_size / 1024**2
        print(f'    \ud83d\udcbe {f.name}: {mb:.1f} MB')
    print()
    print('  \ud83d\udcc1 \u0414\u0430\u043d\u043d\u044b\u0435:  MyDrive/TarsData/')
    print('  \ud83e\udde0 \u041c\u043e\u0434\u0435\u043b\u0438: MyDrive/TarsModels/')
    print('  \ud83d\ude80 \u0417\u0430\u043f\u0443\u0441\u043a:  python launch_tars.py')
else:
    print(f'  \u26a0\ufe0f  \u041e\u0448\u0438\u0431\u043a\u0430 (\u043a\u043e\u0434 {returncode})')
    print(f'     \u0412\u0440\u0435\u043c\u044f: {minutes:.0f} \u043c\u0438\u043d')
    print()
    print('  \u041c\u043e\u0434\u0435\u043b\u0438 \u0441\u043e\u0445\u0440\u0430\u043d\u0435\u043d\u044b \u043d\u0430 Drive.')
    print('  \u23ef\ufe0f  \u041f\u0440\u043e\u0441\u0442\u043e \u043f\u0435\u0440\u0435\u0437\u0430\u043f\u0443\u0441\u0442\u0438\u0442\u0435 \u044d\u0442\u0443 \u044f\u0447\u0435\u0439\u043a\u0443 \u2014 \u043e\u0431\u0443\u0447\u0435\u043d\u0438\u0435 \u043f\u0440\u043e\u0434\u043e\u043b\u0436\u0438\u0442\u0441\u044f.')
    print('  \ud83d\udccb \u041b\u043e\u0433\u0438: local_train.log')
print('=' * 60)

In [ ]:
#@title 📊 Статус обучения { display-mode: "form" }
from pathlib import Path

DRIVE_DATA   = Path('/content/drive/MyDrive/TarsData')
DRIVE_MODELS = Path('/content/drive/MyDrive/TarsModels')

print('=' * 50)
print('  \ud83d\udcca \u0421\u0442\u0430\u0442\u0443\u0441 \u043e\u0431\u0443\u0447\u0435\u043d\u0438\u044f')
print('=' * 50)

if DRIVE_DATA.exists():
    txts = list(DRIVE_DATA.glob('*.txt'))
    total = sum(f.stat().st_size for f in txts) / 1024**2
    print(f'  \ud83d\udcc1 \u0414\u0430\u043d\u043d\u044b\u0435: {len(txts)} \u0444\u0430\u0439\u043b\u043e\u0432, {total:.0f} MB')

if DRIVE_MODELS.exists():
    for f in sorted(DRIVE_MODELS.rglob('*.pt')):
        mb = f.stat().st_size / 1024**2
        print(f'  \ud83e\udde0 {f.relative_to(DRIVE_MODELS)}: {mb:.1f} MB')

try:
    import torch
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f'  \ud83c\udfae VRAM: {alloc:.1f}/{total:.1f} GB')
except: pass

print()
print('  \ud83d\udccb \u041f\u043e\u0441\u043b\u0435\u0434\u043d\u0438\u0435 \u0441\u0442\u0440\u043e\u043a\u0438 \u043b\u043e\u0433\u0430:')
log = Path('/content/TarsSSM-Py/local_train.log')
if log.exists():
    lines = log.read_text(encoding='utf-8', errors='replace').strip().split('\n')
    for l in lines[-15:]:
        print(f'    {l}')
else:
    print('    (\u043d\u0435\u0442 \u043b\u043e\u0433\u0430)')
print('=' * 50)

In [ ]:
#@title 📥 Скачать модели локально { display-mode: "form" }
from google.colab import files
from pathlib import Path
import zipfile, os

DRIVE_MODELS = Path('/content/drive/MyDrive/TarsModels')
tars_v3 = DRIVE_MODELS / 'tars_v3'

if tars_v3.exists():
    zip_path = '/content/tars_v3_models.zip'
    print('\ud83d\udce6 \u0423\u043f\u0430\u043a\u043e\u0432\u043a\u0430...')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in tars_v3.rglob('*'):
            if f.is_file():
                zf.write(f, f.relative_to(DRIVE_MODELS))
                print(f'  + {f.name} ({f.stat().st_size / 1024**2:.1f} MB)')
    
    total_mb = os.path.getsize(zip_path) / 1024**2
    print(f'\n\ud83d\udcbe \u0418\u0442\u043e\u0433\u043e: {total_mb:.0f} MB')
    print('\ud83d\udce5 \u0421\u043a\u0430\u0447\u0438\u0432\u0430\u043d\u0438\u0435...')
    files.download(zip_path)
else:
    print('\u26a0\ufe0f  \u041c\u043e\u0434\u0435\u043b\u0438 \u043d\u0435 \u043d\u0430\u0439\u0434\u0435\u043d\u044b. \u0421\u043d\u0430\u0447\u0430\u043b\u0430 \u0437\u0430\u043f\u0443\u0441\u0442\u0438\u0442\u0435 \u043e\u0431\u0443\u0447\u0435\u043d\u0438\u0435.')

### 📝 Полезные команды
```python
# Обучить только Mamba-2 (фаза 4)
!cd /content/TarsSSM-Py && python -u local_train.py --level medium --drive colab --phase 4 --resume

# Только CoT (фаза 12)
!cd /content/TarsSSM-Py && python -u local_train.py --level medium --drive colab --phase 12 --resume

# Быстрый тест
!cd /content/TarsSSM-Py && python -u local_train.py --level small --drive colab --resume

# MAX модель
!cd /content/TarsSSM-Py && python -u local_train.py --level max --drive colab --resume

# MARATHON (4 дня, макс. качество)
!cd /content/TarsSSM-Py && python -u local_train.py --level marathon --drive colab --resume

# Посмотреть лог
!tail -30 /content/TarsSSM-Py/local_train.log
```